# Word2Vec using Gensim

In [1]:
import subprocess
import sys
import os
import csv

# ---------------- INSTALL LIBRARIES ----------------
def maintain_dependencies():
    required_libraries = ['numpy', 'scipy', 'gensim']

    for lib in required_libraries:
        try:
            __import__(lib)
        except ImportError:
            print(f"Installing {lib}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

maintain_dependencies()

# ---------------------------------------------------

import numpy as np
from scipy.stats import spearmanr, pearsonr
from gensim.models import FastText


# ---------------- COSINE SIMILARITY ----------------
def cosine_similarity(vec1, vec2):

    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot / (norm1 * norm2)


# ---------------- LOAD CORPUS ----------------
def load_text_file(filepath):

    sentences = []

    try:
        with open(filepath, 'r', encoding='utf-8') as f:

            for line in f:

                tokens = line.lower().strip().split()

                if tokens:
                    sentences.append(tokens)

    except Exception as e:
        print("Error loading corpus:", e)

    return sentences


# ---------------- LOAD TEST PAIRS ----------------
def load_test_pairs(filepath):

    pairs = []

    try:
        with open(filepath, 'r', encoding='utf-8') as f:

            reader = csv.reader(f)
            header = next(reader)
            print("CSV columns detected:", header)  # sanity check

            for row in reader:

                if len(row) < 6:
                    continue

                w1 = row[0].strip().strip('"').strip(',')
                w2 = row[1].strip().strip('"').strip(',')

                try:
                    score = float(row[5].strip())  # heuristic_score column
                except ValueError:
                    continue

                pairs.append((w1, w2, score))

        print(f"Loaded {len(pairs)} pairs")

    except Exception as e:
        print("Error loading test pairs:", e)

    return pairs


# ---------------- CLASSIFICATION METRICS ----------------
def confusion_matrix_np(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    return tn, fp, fn, tp


def accuracy_np(tp, tn, fp, fn):

    total = tp + tn + fp + fn
    return (tp + tn) / total if total > 0 else 0


def precision_np(tp, fp):

    return tp / (tp + fp) if (tp + fp) > 0 else 0


def recall_np(tp, fn):

    return tp / (tp + fn) if (tp + fn) > 0 else 0


def f1_np(p, r):

    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0


# ---------------- MAIN ----------------
if __name__ == "__main__":

    CORPUS_FILE = "isizulu_corpus.txt"
    TEST_FILE   = "isizulu_heuristic_5000_pairs.csv"
    OUTPUT_FILE = "fasttext_results.csv"


    # ---- Load Corpus ----
    print("\nLoading corpus...")
    sentences = load_text_file(CORPUS_FILE)

    if not sentences:
        print("Corpus empty — check that the file exists and has content.")
        sys.exit()

    print("Sentences loaded:", len(sentences))
    total_tokens = sum(len(s) for s in sentences)
    print("Total tokens in corpus:", total_tokens)

    if total_tokens < 100_000:
        print("WARNING: Corpus is very small. FastText needs a large corpus for good vectors.")
        print("         Consider using a larger corpus (ideally 1M+ tokens) for better results.")


    # ---- Train FastText ----
    print("\nTraining FastText model...")

    model = FastText(
        sentences=sentences,
        vector_size=300,
        window=5,
        min_count=2,
        sg=1,
        epochs=10,
        workers=4,
        min_n=2,
        max_n=10,
        alpha=0.05,
        sample=1e-1


    )

    print("Vocabulary size:", len(model.wv))

    if len(model.wv) < 100:
        print("WARNING: Very small vocabulary. Check that your corpus is tokenised correctly.")


    # ---- Load Test Pairs ----
    print("\nLoading heuristic pairs...")
    isi_test_pairs = load_test_pairs(TEST_FILE)

    if not isi_test_pairs:
        print("No pairs loaded — check that the CSV exists and has the expected format.")
        sys.exit()


    # ---- Compute Similarities ----
    cosine_scores = []
    human_scores  = []
    results       = []
    skipped       = 0
    skip_reasons  = {}

    print("\nCalculating similarities...\n")

    for i, (w1, w2, hscore) in enumerate(isi_test_pairs):

        if i % 500 == 0:
            print(f"Processing {i} / {len(isi_test_pairs)}")

        try:
            vec1 = model.wv[w1]
            vec2 = model.wv[w2]
            cos  = cosine_similarity(vec1, vec2)

            cosine_scores.append(cos)
            human_scores.append(hscore)

            results.append({
                "word1":             w1,
                "word2":             w2,
                "human_score":       hscore,
                "cosine_similarity": cos
            })

        except Exception as e:
            reason = type(e).__name__
            skip_reasons[reason] = skip_reasons.get(reason, 0) + 1
            skipped += 1

    print(f"\nPairs used   : {len(cosine_scores)}")
    print(f"Pairs skipped: {skipped}")

    if skip_reasons:
        print("Skip reasons:", skip_reasons)


    # ---- Diagnostics ----
    print("\n--- Diagnostics ---")
    print("Cosine score sample (first 10):", [round(c, 4) for c in cosine_scores[:10]])
    print("Human score sample  (first 10):", [round(h, 4) for h in human_scores[:10]])

    cosine_std = np.std(cosine_scores) if cosine_scores else 0
    human_std  = np.std(human_scores)  if human_scores  else 0

    print(f"Cosine std dev : {cosine_std:.6f}")
    print(f"Human  std dev : {human_std:.6f}")
    print(f"Unique cosine values : {len(set(round(c, 6) for c in cosine_scores))}")
    print(f"Cosine min / max : {min(cosine_scores):.4f} / {max(cosine_scores):.4f}" if cosine_scores else "No cosine scores")
    print(f"Human  min / max : {min(human_scores):.4f} / {max(human_scores):.4f}"  if human_scores  else "No human scores")
    print("-------------------\n")


    # ---- Guard: enough pairs? ----
    if len(cosine_scores) < 2:
        print("ERROR: Not enough valid pairs to compute correlation (need at least 2).")
        print("       Check that word spellings in the CSV match the corpus vocabulary.")
        sys.exit()

    # ---- Guard: zero variance? ----
    if cosine_std == 0:
        print("ERROR: All cosine similarity scores are identical — correlation is undefined (NaN).")
        print("       This usually means FastText is returning near-identical subword vectors for all pairs.")
        print("       Possible fixes:")
        print("         1. Use a larger corpus so the model learns meaningful word vectors.")
        print("         2. Check that your corpus vocabulary overlaps with the test pair words.")
        print("         3. Verify corpus tokenisation matches the test pair word forms.")
        sys.exit()

    if human_std == 0:
        print("ERROR: All human scores are identical — correlation is undefined (NaN).")
        print("       Check the label distribution in your CSV.")
        sys.exit()


    # ---- Correlation ----
    rho,  rho_p  = spearmanr(human_scores, cosine_scores)
    pear, pear_p = pearsonr(human_scores, cosine_scores)

    print("Correlation Results")
    print(f"Spearman rho  : {rho:.4f}  (p = {rho_p:.4e})")
    print(f"Pearson  r    : {pear:.4f}  (p = {pear_p:.4e})")


    # ---- Classification ----
    human_median  = np.median(human_scores)
    cosine_median = np.median(cosine_scores)

    y_true = (np.array(human_scores)  >= human_median).astype(int)
    y_pred = (np.array(cosine_scores) >= cosine_median).astype(int)

    tn, fp, fn, tp = confusion_matrix_np(y_true, y_pred)

    accuracy  = accuracy_np(tp, tn, fp, fn)
    precision = precision_np(tp, fp)
    recall    = recall_np(tp, fn)
    f1        = f1_np(precision, recall)

    print("\nClassification Metrics")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1        : {f1:.4f}")

    print("\nConfusion Matrix")
    print(f"  TP={tp}  FP={fp}")
    print(f"  FN={fn}  TN={tn}")


    # ---- Save Results CSV ----
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:

        writer = csv.DictWriter(f, fieldnames=[
            "word1", "word2", "human_score", "cosine_similarity"
        ])
        writer.writeheader()

        for r in results:
            writer.writerow(r)

    print(f"\nResults saved to {OUTPUT_FILE}")


    # ---- Save Metrics ----
    metrics_file = "evaluation_metrics.txt"

    with open(metrics_file, 'w') as f:

        f.write("ISI ZULU FASTTEXT EVALUATION\n")
        f.write("=" * 40 + "\n\n")

        f.write("CORPUS STATS\n")
        f.write(f"Sentences : {len(sentences)}\n")
        f.write(f"Tokens    : {total_tokens}\n")
        f.write(f"Vocab size: {len(model.wv)}\n\n")

        f.write("PAIR STATS\n")
        f.write(f"Pairs used   : {len(cosine_scores)}\n")
        f.write(f"Pairs skipped: {skipped}\n\n")

        f.write("DIAGNOSTICS\n")
        f.write(f"Cosine std dev : {cosine_std:.6f}\n")
        f.write(f"Human  std dev : {human_std:.6f}\n\n")

        f.write("CORRELATION\n")
        f.write(f"Spearman rho : {rho:.4f}  (p = {rho_p:.4e})\n")
        f.write(f"Pearson  r   : {pear:.4f}  (p = {pear_p:.4e})\n\n")

        f.write("CLASSIFICATION\n")
        f.write(f"Accuracy  : {accuracy:.4f}\n")
        f.write(f"Precision : {precision:.4f}\n")
        f.write(f"Recall    : {recall:.4f}\n")
        f.write(f"F1        : {f1:.4f}\n\n")

        f.write("CONFUSION MATRIX\n")
        f.write(f"  TP={tp}  FP={fp}\n")
        f.write(f"  FN={fn}  TN={tn}\n")

    print(f"Metrics saved to {metrics_file}")
    print("\nEvaluation complete!")


Loading corpus...
Sentences loaded: 91710
Total tokens in corpus: 1767026

Training FastText model...
Vocabulary size: 6371

Loading heuristic pairs...
CSV columns detected: ['word1', 'word2', 'strategy', 'detail', 'cosine_similarity', 'heuristic_score', 'in_vocab']
Loaded 4332 pairs

Calculating similarities...

Processing 0 / 4332
Processing 500 / 4332
Processing 1000 / 4332
Processing 1500 / 4332
Processing 2000 / 4332
Processing 2500 / 4332
Processing 3000 / 4332
Processing 3500 / 4332
Processing 4000 / 4332

Pairs used   : 4332
Pairs skipped: 0

--- Diagnostics ---
Cosine score sample (first 10): [np.float32(0.5806), np.float32(0.1861), np.float32(0.5408), np.float32(0.2446), np.float32(0.6845), np.float32(0.4845), np.float32(0.6569), np.float32(0.625), np.float32(0.4228), np.float32(0.6079)]
Human score sample  (first 10): [0.443, 0.2227, 0.4866, 0.3836, 0.5698, 0.5188, 0.5624, 0.273, 0.1574, 0.4643]
Cosine std dev : 0.150610
Human  std dev : 0.135966
Unique cosine values : 4314